# Dust Optical Depth (550 nm) from CAMS: daily series over Geneva

Alternative source to the BSC notebook. Data: **Copernicus Atmosphere Monitoring Service (CAMS)**, variable *Dust aerosol optical depth at 550 nm* (`duaod550`), downloaded from the Atmosphere Data Store (ADS) with `cdsapi`.

Two CAMS datasets are supported (choose in CONFIG):

| `CAMS_DATASET` | ADS dataset | Notes |
|---|---|---|
| `"reanalysis"` | `cams-global-reanalysis-eac4` | 3-hourly, ~0.75° grid, consistent over 2003 to near present. **Recommended for historical years.** |
| `"forecast"` | `cams-global-atmospheric-composition-forecasts` | 00 UTC run, lead times 0 to 21 h (3-hourly), ~0.4° grid. Use for recent periods the reanalysis does not cover yet. |

**Workflow**: CONFIG, download month by month (cached), spatial reduction to the Geneva bbox, daily mean, export CSV and plot.

**Things to know**
- CAMS is a **model** (like BSC/MONARCH), with assimilation of total AOD. Dust AOD itself is a model-derived field.
- The grid (0.4 to 0.75°, i.e. 40 to 80 km) is much coarser than the Geneva bbox. The default `SPATIAL_METHOD = "bilinear_centroid"` interpolates the field to the bbox centre.
- **Credentials**: register on https://ads.atmosphere.copernicus.eu, accept the licence of the dataset, then either put the key in `~/.cdsapirc` or set the environment variables `ADS_URL` and `ADS_KEY`.

In [ ]:
# ============================== CONFIG ==============================
DEMO_MODE = False            # True = synthetic series, no network / no credentials needed

START_DATE = "2025-01-01"    # inclusive
END_DATE   = "2025-12-31"    # inclusive

# Geneva bounding box (decimal degrees, WGS84): replace with the one used in the BSC notebook
BBOX = {"west": 5.95, "south": 46.13, "east": 6.31, "north": 46.37}

CAMS_DATASET   = "reanalysis"          # "reanalysis" (EAC4) or "forecast"
SPATIAL_METHOD = "bilinear_centroid"   # or "mean_in_bbox" (mean of grid cells whose centre is in the bbox)
GRID_PAD_DEG   = 1.0                   # padding around the bbox for the download area

import os
ADS_URL = os.environ.get("ADS_URL", "https://ads.atmosphere.copernicus.eu/api")
ADS_KEY = os.environ.get("ADS_KEY", "ADD_YOUR_KEY_HERE")    # None = read ~/.cdsapirc

OUTPUT_DIR = "outputs_dod_cams"

In [2]:
#%pip install cdsapi xarray netCDF4 scipy pandas matplotlib
import os, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

assert CAMS_DATASET in ("reanalysis", "forecast"), "CAMS_DATASET must be 'reanalysis' or 'forecast'"
assert SPATIAL_METHOD in ("bilinear_centroid", "mean_in_bbox")
assert pd.Timestamp(START_DATE) <= pd.Timestamp(END_DATE), "START_DATE after END_DATE"

DATASETS = {
    "reanalysis": {
        "name": "cams-global-reanalysis-eac4",
        "times": [f"{h:02d}:00" for h in range(0, 24, 3)],
        "extra": {},
    },
    "forecast": {
        "name": "cams-global-atmospheric-composition-forecasts",
        "times": ["00:00"],
        "extra": {"type": ["forecast"], "leadtime_hour": [str(h) for h in range(0, 24, 3)]},
    },
}
CFG = DATASETS[CAMS_DATASET]
VARIABLE = "dust_aerosol_optical_depth_550nm"

OUT = Path(OUTPUT_DIR)
RAW = OUT / "raw"
RAW.mkdir(parents=True, exist_ok=True)
DAILY_CSV = OUT / f"dod550_cams_{CAMS_DATASET}_geneva_{START_DATE}_{END_DATE}.csv"        # same structure as SDE2022_mean.csv
DIAG_CSV  = OUT / f"dod550_cams_{CAMS_DATASET}_geneva_{START_DATE}_{END_DATE}_diagnostics.csv"  # daily max, n_timesteps

if not DEMO_MODE:
    import cdsapi

## 1. Download (one request per month, cached)

In [ ]:
def month_chunks(start, end):
    """Yield (first_day, last_day) per calendar month, clipped to [start, end]."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    for ms in pd.date_range(start.replace(day=1), end, freq="MS"):
        me = ms + pd.offsets.MonthEnd(0)
        yield max(ms, start), min(me, end)


def download_chunk(client, d0, d1):
    target = RAW / f"cams_{CAMS_DATASET}_{d0:%Y-%m-%d}_{d1:%Y-%m-%d}.nc"
    if target.exists() and target.stat().st_size > 0:
        return target
    area = [BBOX["north"] + GRID_PAD_DEG, BBOX["west"] - GRID_PAD_DEG,
            BBOX["south"] - GRID_PAD_DEG, BBOX["east"] + GRID_PAD_DEG]   # N, W, S, E
    request = {
        "variable": [VARIABLE],
        "date": [f"{d0:%Y-%m-%d}/{d1:%Y-%m-%d}"],
        "time": CFG["times"],
        "data_format": "netcdf",
        "area": area,
        **CFG["extra"],
    }
    tmp = target.with_suffix(".download")
    client.retrieve(CFG["name"], request, str(tmp))
    if zipfile.is_zipfile(tmp):                      # the ADS sometimes returns a zip
        with zipfile.ZipFile(tmp) as z:
            nc = [n for n in z.namelist() if n.endswith(".nc")][0]
            target.write_bytes(z.read(nc))
        tmp.unlink()
    else:
        tmp.rename(target)
    return target


files = []
if DEMO_MODE:
    print("DEMO_MODE: skipping download")
else:
    client = cdsapi.Client(url=ADS_URL, key=ADS_KEY)
    for d0, d1 in month_chunks(START_DATE, END_DATE):
        print(f"{d0:%Y-%m-%d} -> {d1:%Y-%m-%d} ...", end=" ")
        files.append(download_chunk(client, d0, d1))
        print("ok")

## 2. Spatial reduction to the Geneva bbox and conversion to a time series

In [ ]:
def _name(ds, candidates):
    for c in candidates:
        if c in ds.variables or c in ds.dims:
            return c
    raise KeyError(f"None of {candidates} in {list(ds.variables)}")


def file_to_series(path):
    ds = xr.open_dataset(path)
    try:
        var = "duaod550" if "duaod550" in ds.data_vars else list(ds.data_vars)[0]
        da = ds[var].squeeze(drop=False)
        latn = _name(ds, ["latitude", "lat"])
        lonn = _name(ds, ["longitude", "lon"])
        # CAMS global fields can use 0..360 longitudes; Geneva is at +6 deg so this is only a safeguard
        if float(da[lonn].max()) > 180 and BBOX["west"] < 0:
            da = da.assign_coords({lonn: ((da[lonn] + 180) % 360) - 180}).sortby(lonn)

        if SPATIAL_METHOD == "bilinear_centroid":
            clat = 0.5 * (BBOX["south"] + BBOX["north"])
            clon = 0.5 * (BBOX["west"] + BBOX["east"])
            red = da.interp({latn: clat, lonn: clon}, method="linear")
        else:
            sel = da.where(
                (da[latn] >= BBOX["south"]) & (da[latn] <= BBOX["north"]) &
                (da[lonn] >= BBOX["west"]) & (da[lonn] <= BBOX["east"]), drop=True)
            if sel[latn].size == 0 or sel[lonn].size == 0:
                raise ValueError("No CAMS grid-cell centre falls inside the bbox; use 'bilinear_centroid'.")
            red = sel.mean(dim=[latn, lonn], skipna=True)

        # valid time: 'valid_time' (new ADS files) or 'time'
        tname = "valid_time" if "valid_time" in red.coords else ("time" if "time" in red.coords else None)
        if tname is None:
            raise KeyError(f"No time coordinate found: {list(red.coords)}")
        vt, vals = xr.broadcast(red[tname], red.drop_vars([c for c in red.coords if c not in red.dims and c != tname]))
        vt = vt.transpose(*vals.dims)
        s = pd.Series(np.asarray(vals.values, dtype="float64").ravel(),
                      index=pd.to_datetime(np.asarray(vt.values).ravel()))
        return s[~s.index.duplicated()].sort_index()
    finally:
        ds.close()


def make_demo_series(start, end, seed=7):
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, pd.Timestamp(end) + pd.Timedelta(hours=21), freq="3h")
    base = rng.lognormal(np.log(0.015), 0.5, len(idx))
    events = np.zeros(len(idx))
    for _ in range(max(1, len(idx) // 900)):              # a few Saharan dust episodes
        i = rng.integers(0, len(idx) - 40)
        events[i:i + rng.integers(8, 40)] += rng.uniform(0.1, 0.5)
    return pd.Series(base + events, index=idx)


if DEMO_MODE:
    series = make_demo_series(START_DATE, END_DATE)
else:
    series = pd.concat([file_to_series(f) for f in files]).sort_index()
    series = series[~series.index.duplicated()]

print(f"{len(series)} time steps, {series.index.min()} -> {series.index.max()}")
series.head()

## 3. Daily aggregation (UTC days)

In [ ]:
grp = series.resample("D")
daily = pd.DataFrame({
    "dod_550": grp.mean(),
    "dod_550_max": grp.max(),
    "n_timesteps": grp.count(),
})
daily.index.name = "date"
daily = daily.loc[START_DATE:END_DATE]
daily = daily.reindex(pd.date_range(START_DATE, END_DATE, freq="D", name="date"))
daily["n_timesteps"] = daily["n_timesteps"].fillna(0).astype(int)

expected = len(CFG["times"]) if CAMS_DATASET == "reanalysis" else 8
incomplete = daily[(daily["n_timesteps"] != expected)]
print(f"{len(daily)} days; {len(incomplete)} days with != {expected} time steps")
daily.describe()

## 4. Export and plot

Main CSV has the same columns as the BSC output (`time, latitude, longitude, OD550_DUST`); latitude/longitude are the bbox centre. Extra columns go to a separate `_diagnostics.csv`.

In [ ]:
def write_reference_csv(series, path):
    """Write time,latitude,longitude,OD550_DUST (one row per day; missing days as 'date,,,')."""
    lat_c = 0.5 * (BBOX["south"] + BBOX["north"])
    lon_c = 0.5 * (BBOX["west"] + BBOX["east"])
    out = pd.DataFrame({"time": series.index.strftime("%Y-%m-%d"),
                        "latitude": lat_c, "longitude": lon_c,
                        "OD550_DUST": series.values})
    out.loc[out["OD550_DUST"].isna(), ["latitude", "longitude"]] = np.nan
    out.to_csv(path, index=False, float_format="%.8g")


write_reference_csv(daily["dod_550"], DAILY_CSV)
daily.to_csv(DIAG_CSV, float_format="%.5f")
print("Saved", DAILY_CSV, "and", DIAG_CSV)

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(daily.index, daily["dod_550"], color="tab:orange", alpha=0.3)
ax.plot(daily.index, daily["dod_550"], color="tab:orange", lw=1.2, label="Daily mean DOD 550 nm")
ax.plot(daily.index, daily["dod_550_max"], color="0.6", lw=0.6, label="Daily max (3-hourly steps)")
ax.set_ylabel("Dust optical depth (-)")
ax.set_title(f"CAMS ({CAMS_DATASET}) DOD at 550 nm, Geneva" + (" [DEMO DATA]" if DEMO_MODE else ""))
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUT / f"dod550_cams_{CAMS_DATASET}_geneva_{START_DATE}_{END_DATE}.png", dpi=150)
plt.show()